# HTR pilot: text rules, split, tokenisation, decoding

Stage 18–19 Oct (recognition). This notebook fixes how raw transcripts become recognition targets,
creates the provisional train/validation split and ground-truth handwritten line crops, checks how
candidate TrOCR checkpoints tokenise and decode Ukrainian, and runs a short training pilot to size
the full run in `02_htr_baseline_train.ipynb`.

Only the official `train` split is used for crops and models. `test` appears once below, for marker counts only.

In [1]:
import math
import sys
import time
from collections import Counter
from multiprocessing import Pool

import pandas as pd
import torch
from PIL import Image
from transformers import TrOCRProcessor, VisionEncoderDecoderModel, get_linear_schedule_with_warmup
from transformers.utils import logging

sys.path.insert(0, '../src')
from htr import (CROP_DIR, CROP_FILE, SOURCES, SPLIT_FILE, STRIKE, LineDataset, cer, clean_text, collate,
                 is_htr_line, load_pages, predict, snapshot, train_epoch, wer)

logging.set_verbosity_error()
logging.disable_progress_bar()
pd.set_option('display.max_colwidth', 120)
SEED = 42

/home/stalex/unik_shit/ai/pandemonium-rukopys/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Text markers

HTR lines are legible, Ukrainian, non-empty `handwritten` regions (same definition as `data/eda.md`).
The dataset card defines `~~struck~~`, `~~old~~{new}`, and `[illegible]`. A few lines use single
`~` pairs, which are probably malformed strikethroughs.

In [2]:
lines = pd.DataFrame([dict(split=split, source=page['source'], text=region['text'])
                      for split in ('train', 'test') for page in load_pages(split)
                      for region in page['regions'] if is_htr_line(region)])
lines['target'] = lines.text.map(clean_text)
text = lines.text
correction = text.str.contains(r'~~.*?~~\{')
pd.DataFrame({
    'lines': 1,
    '[illegible]': text.str.contains('[illegible]', regex=False),
    '~~old~~{new}': correction,
    '~~struck~~': text.str.contains('~~', regex=False) & ~correction,
    'unmatched ~': text.str.replace(STRIKE, '', regex=True).str.contains('~', regex=False),
    'line break': text.str.contains('\n', regex=False),
    'skipped': lines.target.isna(),
}).groupby([lines.split, lines.source]).sum()

lines  [illegible]  ~~old~~{new}  ~~struck~~  unmatched ~  \
split source                                                                  
test  archive       509            1             0          15            0   
      dictation    3397            2            17          67            2   
      school       1238            4             0          31            2   
      university   1203            8             5          33            0   
train archive      2454            9             4          45            0   
      dictation    8383           17             9          56            6   
      school       7995           11            34         161            2   
      university   2538           23            14          66            1   

                  line break  skipped  
split source                           
test  archive              0        1  
      dictation            0        9  
      school               0        8  
      university           0       12  
train archive              2       10  
      dictation            2       24  
      school               8       21  
      university           0       30

Rules used for recognition targets (`clean_text` in `src/htr.py`). Raw annotations stay unchanged;
the target is stored next to the raw text in the crop table.

- `~~old~~{new}` → `new`, `~~struck~~` → removed. The model should read what the writer meant, not what was crossed out.
- Lines with `[illegible]` or unmatched `~` are skipped for training and scoring: the target does not describe the image.
- Unicode NFC, apostrophe variants (`’ ʼ ‘`) → `'`, line breaks and repeated spaces → one space.
- Everything else is kept as written, including Latin, math symbols, and old/Russian letters in archive pages.

In [3]:
changed = lines[(lines.split == 'train') & (lines.text != lines.target)]
changed.sample(10, random_state=SEED)[['source', 'text', 'target']]

,source,text,target
18394,school,"Урок позакласного читання. К. Буличов."" Ц~~к~~икл ""Аліса"".","Урок позакласного читання. К. Буличов."" Цикл ""Аліса""."
11527,university,"цих відносин є валютні ринки, на яких здійснює~~с~~ться","цих відносин є валютні ринки, на яких здійснюється"
18955,school,6. Ні. ~~т~~ Потрібно не йти а переходити,6. Ні. Потрібно не йти а переходити
16565,school,"вона швидше замерзає (+16, 6 °C), і ~~тот~~","вона швидше замерзає (+16, 6 °C), і"
12869,university,"США: ~~ВВП~~ = ЗТО = 1277 + 2180 = 3457; Експортна квота = ( 1277 / 24330 ) * 100 % = 5,25 %","США: = ЗТО = 1277 + 2180 = 3457; Експортна квота = ( 1277 / 24330 ) * 100 % = 5,25 %"
2858,dictation,радіо доносило і в сибірську пущу,радіо доносило і в сибірську пущу
20048,school,обов'язок\nслужба в кінноті,обов'язок служба в кінноті
10605,archive,всяку охоту в козаків та старшинах до ~~таких~~тих,всяку охоту в козаків та старшинах до тих
20511,school,"Скажи ж мені; Дніпре, які вітровії все","Скажи ж мені; Дніпре, які вітровії все"
20046,school,права \nучасть у Раді чотирьохсот; обіймати найвищі держпосади,права участь у Раді чотирьохсот; обіймати найвищі держпосади


In [4]:
train_targets = lines.query("split == 'train'").target.dropna()
ukrainian = set('абвгґдеєжзиіїйклмнопрстуфхцчшщьюя')
ukrainian |= {c.upper() for c in ukrainian}
chars = Counter(''.join(train_targets))
other = {c: n for c, n in chars.items() if c not in ukrainian and not c.isdigit() and c not in ' .,;:!?-()"\''}
print(f'{len(chars)} distinct characters; {sum(other.values()) / sum(chars.values()):.2%} outside Ukrainian letters, digits, basic punctuation')
print('  '.join(f'{c} {n}' for c, n in Counter(other).most_common(40)))

247 distinct characters; 1.06% outside Ukrainian letters, digits, basic punctuation
„ 557  = 515  a 301  “ 261  I 245  / 221  C 204  x 193  A 184  № 173  + 159  — 158  B 155  O 150  l 135  H 128  N 119  % 118  [ 110  ] 105  → 101  i 99  n 96  b 95  y 82  e 81  s 79  S 75  ^ 70  p 61  { 61  * 60  V 57  | 57  } 57  c 55  o 53  • 53  R 52  · 52


## Provisional split

About 10% of official training pages go to validation, sampled per source with seed 42. All lines from a
page stay together. The metadata has no writer or document IDs, and there are no exact duplicate
images in train (`data/eda.md`), so nothing more can be grouped here. **This split still needs to be
agreed with the CV side**; if it changes, rerun this notebook and `02_htr_baseline_train.ipynb`.

In [5]:
pages = pd.DataFrame(load_pages('train'))
val_pages = pages.groupby('source', group_keys=False).sample(frac=0.1, random_state=SEED)
pages['split'] = 'train'
pages.loc[val_pages.index, 'split'] = 'val'
SPLIT_FILE.parent.mkdir(parents=True, exist_ok=True)
pages[['file_name', 'source', 'source_detail', 'annotation_source', 'split']].to_csv(SPLIT_FILE, index=False)
pd.crosstab(pages.source, pages.split, margins=True)

split,train,val,All
source,,,
archive,114,13,127
dictation,323,36,359
school,614,68,682
university,146,16,162
All,1197,133,1330


## Ground-truth line crops

Boxes are cut from raw pixels. The 27 train pages flagged in the EDA have EXIF orientation `0`
(undefined, i.e. no rotation), and their boxes line up with the raw image, so no transpose is applied.
Crops are downscaled to at most 1024 px per side (TrOCR resizes to 384×384 anyway). No padding;
padding is a separate CV experiment.

In [6]:
def crop_page(page):
    page_id = page['file_name'].split('/')[-1].removesuffix('.jpg')
    records = []
    with Image.open(snapshot() / 'train' / page['file_name']) as image:
        image = image.convert('RGB')
        for index, region in enumerate(page['regions']):
            if not is_htr_line(region):
                continue
            crop_path = f'{page_id}_{index:03d}.jpg'
            crop = image.crop(region['bbox'])
            crop.thumbnail((1024, 1024))
            crop.save(CROP_DIR / crop_path, quality=95)
            records.append(dict(page_id=page_id, region=index, bbox=region['bbox'], type=region['type'],
                                crop_path=crop_path, text=region['text'], target=clean_text(region['text']),
                                source=page['source'], annotation_source=page['annotation_source'],
                                split=page['split']))
    return records


CROP_DIR.mkdir(parents=True, exist_ok=True)
with Pool(6) as pool:
    crops = pd.DataFrame([r for rs in pool.imap(crop_page, pages.to_dict('records'), chunksize=4) for r in rs])
crops.to_csv(CROP_FILE, index=False)
crops.groupby(['split', 'source']).agg(lines=('target', 'size'), skipped=('target', lambda t: t.isna().sum()))

lines  skipped
split source                    
train archive      2147        8
      dictation    7579       22
      school       7302       19
      university   2256       26
val   archive       307        2
      dictation     804        2
      school        693        2
      university    282        4

How much validation text is already seen in training? Dictation pages in train all write the same 2024
dictation, so validation dictation lines are nearly memorisable and their CER will be optimistic.

In [7]:
words = crops.dropna(subset=['target']).assign(words=lambda d: d.target.str.lower().str.split())
seen = {}
for source in SOURCES:
    train_vocab = set(words.query("split == 'train' and source == @source").words.explode())
    val_words = words.query("split == 'val' and source == @source").words.explode()
    val_lines = words.query("split == 'val' and source == @source").target
    train_lines = set(words.query("split == 'train' and source == @source").target)
    seen[source] = dict(val_words_in_train_vocab=val_words.isin(train_vocab).mean(),
                        val_lines_identical_to_a_train_line=val_lines.isin(train_lines).mean())
pd.DataFrame(seen).T.round(3)

,val_words_in_train_vocab,val_lines_identical_to_a_train_line
dictation,0.977,0.542
archive,0.591,0.010
university,0.601,0.050
school,0.604,0.165


## Tokenisation

Candidates are TrOCR-base checkpoints: the English original, a Russian handwriting fine-tune, and a
Ukrainian handwriting fine-tune of the Russian one (MultiHTR Ukrainian data, not RUKOPYS). Checkpoints
that were already trained on RUKOPYS are avoided because they may have seen its test pages.

In [8]:
CANDIDATES = {
    'microsoft/trocr-base-handwritten': 'eaacaf452b06415df8f10bb6fad3a4c11e609406',
    'kazars24/trocr-base-handwritten-ru': '5342fbb29ec56eb677f553738c2fcc2befd6b0ab',
    'cyrillic-trocr/trocr-ukrainian-handwritten': '2f717bb4e44af7e5cabf553df2a9aae06322255e',
}
tokenizers = {name: TrOCRProcessor.from_pretrained(name, revision=rev).tokenizer for name, rev in CANDIDATES.items()}
vocabs = [tok.get_vocab() for tok in tokenizers.values()]
print('identical vocabularies:', all(v == vocabs[0] for v in vocabs))
tok = tokenizers['cyrillic-trocr/trocr-ukrainian-handwritten']

identical vocabularies: True


All three share the English RoBERTa byte-level BPE. Nothing is out of vocabulary, but Cyrillic is
split into one- or two-letter pieces, and some pieces are half a UTF-8 character:

In [9]:
example = 'Магія голосу: їжак, ґанок, м\'ята'
print([tok.decode([i]) for i in tok(example, add_special_tokens=False).input_ids])

['�', '�', 'а', '�', '�', '�', '�', 'я', ' �', '�', 'о�', '�', 'о', 'с', 'у', ':', ' ', '�', '�', '�', '�', 'а', 'к', ',', ' ', '�', '�', 'а', 'н', 'о�', '�', ',', ' �', '�', "'", 'я', 'т', 'а']


In [10]:
n_tokens = train_targets.map(lambda t: len(tok(t).input_ids))
roundtrip = train_targets.map(lambda t: tok.decode(tok(t).input_ids, skip_special_tokens=True) == t)
print(f'tokens per character: {n_tokens.sum() / train_targets.str.len().sum():.2f}')
print(f'exact decode(encode(x)) round trip: {roundtrip.mean():.2%}')
print(f'lines over 64 tokens: {(n_tokens > 64).mean():.2%}, over 128: {(n_tokens > 128).sum()}')
n_tokens.describe(percentiles=[.5, .9, .99]).round(1).to_frame('tokens per line').T

tokens per character: 1.20
exact decode(encode(x)) round trip: 100.00%
lines over 64 tokens: 5.53%, over 128: 5


,count,mean,std,min,50%,90%,99%,max
tokens per line,21285.0,35.9,18.8,3.0,38.0,58.0,80.0,567.0


Tokenisation is lossless, but sequences are long: 64 tokens (a common TrOCR default) would cut 5.5% of
lines, sometimes mid-character. 128 tokens covers all but a handful of lines and is used below.

## Zero-shot comparison

A fixed validation sample of 75 lines per source, greedy decoding, 128-token limit.

In [11]:
val = crops.query("split == 'val'").dropna(subset=['target'])
sample = val.groupby('source', group_keys=False).sample(75, random_state=SEED)
GREEDY = dict(num_beams=1, no_repeat_ngram_size=0, max_length=130)  # start tokens + 128 target tokens


def load(name):
    processor = TrOCRProcessor.from_pretrained(name, revision=CANDIDATES[name])
    model = VisionEncoderDecoderModel.from_pretrained(name, revision=CANDIDATES[name]).cuda()
    return processor, model


def score(predictions, frame=sample):
    return dict(CER=cer(frame.target, predictions), WER=wer(frame.target, predictions))


results = {}
for name in CANDIDATES:
    processor, model = load(name)
    start = time.time()
    hyps = predict(model, processor, sample, **GREEDY)
    results[name] = score(hyps) | {'ms/line': 1000 * (time.time() - start) / len(sample)}
    del model
    torch.cuda.empty_cache()
pd.DataFrame(results).T.round(3)

,CER,WER,ms/line
microsoft/trocr-base-handwritten,1.040,1.272,36.459
kazars24/trocr-base-handwritten-ru,0.621,0.994,53.813
cyrillic-trocr/trocr-ukrainian-handwritten,0.252,0.612,36.871


## Decoding

Same sample, Ukrainian checkpoint. Its model card recommends no repeat-n-gram block and 128 tokens.

In [12]:
processor, model = load('cyrillic-trocr/trocr-ukrainian-handwritten')
settings = {
    'greedy, 128 tokens': GREEDY,
    'greedy, 64 tokens': GREEDY | dict(max_length=64),
    'greedy, no_repeat_ngram_size=3': GREEDY | dict(no_repeat_ngram_size=3),
    'beam 4, 128 tokens': GREEDY | dict(num_beams=4),
}
decoding = {}
for label, kwargs in settings.items():
    start = time.time()
    hyps = predict(model, processor, sample, **kwargs)
    decoding[label] = score(hyps) | {'ms/line': 1000 * (time.time() - start) / len(sample)}
pd.DataFrame(decoding).T.round(3)

,CER,WER,ms/line
"greedy, 128 tokens",0.252,0.612,36.129
"greedy, 64 tokens",0.272,0.620,32.398
"greedy, no_repeat_ngram_size=3",0.265,0.645,40.513
"beam 4, 128 tokens",0.246,0.603,128.777


## Training pilot

1,000 random training lines, two epochs, batch 16, AdamW 3e-5 with linear warm-up/decay, bf16
autocast. Full fine-tuning of TrOCR-base (334M parameters) with AdamW needs gradient checkpointing to fit
in the 8 GB laptop GPU; without it, memory spills into system RAM and a step takes seconds.

In [13]:
del model
torch.cuda.empty_cache()
processor, model = load('cyrillic-trocr/trocr-ukrainian-handwritten')
torch.manual_seed(SEED)
pilot = crops.query("split == 'train'").dropna(subset=['target']).sample(1000, random_state=SEED)
loader = torch.utils.data.DataLoader(LineDataset(pilot, processor), batch_size=16, shuffle=True, num_workers=4,
                                     collate_fn=lambda b: collate(b, processor.tokenizer.pad_token_id))
model.gradient_checkpointing_enable()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5, weight_decay=0.01, fused=True)
scheduler = get_linear_schedule_with_warmup(optimizer, 12, 2 * len(loader))
torch.cuda.reset_peak_memory_stats()
start = time.time()
losses = [train_epoch(model, loader, optimizer, scheduler) for _ in range(2)]
seconds_per_step = (time.time() - start) / (2 * len(loader))
print(f'epoch mean loss: {[round(sum(l) / len(l), 3) for l in losses]}')
print(f'{seconds_per_step:.2f} s/step, peak GPU memory {torch.cuda.max_memory_allocated() / 1e9:.1f} GB')

epoch mean loss: [0.692, 0.308]
0.80 s/step, peak GPU memory 6.0 GB


In [14]:
after = score(predict(model, processor, sample, **GREEDY))
n_train = crops.query("split == 'train'").target.notna().sum()
epoch_minutes = math.ceil(n_train / 16) * seconds_per_step / 60
print(f'full training split: {n_train} lines, about {epoch_minutes:.0f} min per epoch at this speed')
pd.DataFrame({'zero-shot': results['cyrillic-trocr/trocr-ukrainian-handwritten'], 'after 1k-line pilot': after}).T[['CER', 'WER']].round(3)

full training split: 19209 lines, about 16 min per epoch at this speed


,CER,WER
zero-shot,0.252,0.612
after 1k-line pilot,0.241,0.596


## Conclusions

- **Targets:** the rules above skip about 0.4% of HTR lines; everything else, including archive spelling and math symbols, is kept.
- **Tokenisation:** all candidates use the same English byte-level BPE. It is lossless for Ukrainian, at about 1.2 tokens per
  character, so a 128-token limit is needed. Swapping in a Ukrainian tokenizer would shorten sequences but throw away the
  pretrained decoder output layer, so it is not worth doing before the midterm.
- **Checkpoint:** the Ukrainian fine-tune is far ahead zero-shot (CER ≈ 0.25 vs ≈ 0.62 Russian and > 1 English), so it is the
  starting point for baseline A.
- **Decoding:** no repeat-n-gram block and 128 tokens. Beam 4 is slightly better than greedy at about three times the cost, so
  greedy is used for per-epoch model selection and beam 4 for the final validation numbers. The checkpoint config disables
  the KV cache; `predict` turns it back on, which makes generation roughly eight times faster.
- **Pilot:** the model learns from a small subset, fits in 8 GB with gradient checkpointing, and takes about 15 minutes per
  epoch on the full training split. Baseline A therefore trains for 5 epochs (about 1.5 hours with validation).
- **Caveat:** about half of validation dictation lines appear verbatim in training (one shared dictation text), so dictation
  CER is optimistic. Results are reported per source.